In [ ]:
import pickle
import os
import random
import pandas as pd
from string import punctuation

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from nltk.tag import pos_tag
from nltk.probability import FreqDist
from nltk.classify import NaiveBayesClassifier, accuracy

### **Preprocessing**

In [ ]:
eng_stop = stopwords.words('english')
stemmer = SnowballStemmer('english')
lemmatizer = WordNetLemmatizer()

In [ ]:
def AlterTag (tag: str):
    if tag.startswith('J'):
        return 'a'
    elif tag.startswith('V'):
        return 'v'
    elif tag.startswith('R'):
        return 'r'
    return 'n'

def Preprocess(text: str):
    tokens = word_tokenize(text.lower())
    tokens = [tok for tok in tokens if tok.isalpha()]
    tokens = [tok for tok in tokens if tok not in punctuation]
    tokens = [tok for tok in tokens if tok not in eng_stop]
    tokens = [stemmer.stem(tok) for tok in tokens]

    tagged = pos_tag(tokens)

    tokens = [lemmatizer.lemmatize(tok, AlterTag(tag)) for tok, tag in tagged]
    return tokens

### **Training**

In [ ]:
def Training():
    data = pd.read_csv('./Dataset/Tweets.csv')
    X = data['text']
    Y = data['airline_sentiment']

    # Feature Extraction
    feats = []

    for text, label in zip(X, Y):
        clean = Preprocess(text)
        ft = {tok: True for tok in clean}
        feats.append((ft, label))
    
    random.shuffle(feats)

    # Training
    split = int(len(feats) * 0.8)
    train_data = feats[:split]
    evals_data = feats[split:]

    print('Starts Training...')
    model = NaiveBayesClassifier.train(train_data)
    acc = accuracy(model, evals_data)

    print(f'The Accuracy of the Model: {(acc*100):.2f}%')
    print('')

    # Info
    print(f'The Top 5 Most Informative Features are:')
    model.show_most_informative_features(5)
    print('')

    # Save
    with open('./model.pickle', 'wb') as file:
        pickle.dump(model, file)
    print('Model Saved')
    print('')

    return model

def Load():
    if os.path.exists('./model.pickle'):
        with open('./model.pickle', 'rb') as file:
            model = pickle.load(file)
        return model
    else:
        model = Training()
        return model

### **Support Menu**

In [ ]:
def Write_Tweet():
    while True:
        print('Input your Tweet: ', end='')
        docx = input()

        if (docx.split() <= 5):
            print(f'Please Tweet at least 5 words')
        else:
            return docx 

def Analyze_Tweet(docx, model):
    base_tokens = word_tokenize(docx)
    tokens = [tok.lower() for tok in base_tokens]
    tokens = [tok for tok in tokens if tok.isalpha()]
    tokens = [tok for tok in tokens if tok not in punctuation]

    # POS Tag

    # Synonyms & Anton

    # Predict


    return None

### **Menu**

In [ ]:
def Menu():
    model = Load()
    docx = ''

    while True:
        print()
        print('1. Write Tweet')
        print('2. Analyze Tweet')
        print('3. End Session')

        cc = input('>> ')

        if cc == '1':
            docx = Write_Tweet()
        elif cc == '2':
            pass
        elif cc == '3':
            print('Alright, Thank You for using our App!!! :)')
            break
        else:
            print('Invalid Input')

In [ ]:
Menu()